# 17th ACM Conference on Bioinformatics, Computational Biology, and Health Informatics (ACM-BCB 2026), University of Calabria.

## T1: Spatial transcriptomics meets advanced image analysis: AI-driven integration of spatial omics data
**Authors:** Sara Terzoli & Matteo Bonfanti \
*National Facility for Data Handling and Analysis, Human Technopole, Milan, Italy.*

In this hands-on tutorial, we present a Visium HD analysis workflow integrating spatial transcriptomics data with image-based cell segmentation to infer cell identities and investigate their spatial distribution within the tissue.

The notebook is organized in sequential sections so participants can run one block at a time during the live tutorial.

# Bin2Cell: from bins to cell-level matrix

In this notebook, we start from the **2 µm bin-level expression matrix** generated by Space Ranger for Visium HD data and use **Bin2Cell** to aggregate transcript counts into individual cells.


The goal is to move from a bin-level expression matrix, which is the native representation of Visium HD data, to a cell-level matrix where each observation corresponds to a **segmented cell**. This transformation enables downstream analyses that are more similar to conventional single-cell RNA-seq workflows, including cell-level QC, normalization, dimensionality reduction, clustering, and cell type annotation.

> Important note: The cell segmentation used in this notebook was generated in the previous tutorial and is assumed to be already available. The segmentation mask is provided as a  **.tif image**, where each pixel contains the identifier of the cell to which it belongs. Therefore, this notebook does not cover the segmentation procedure itself. Instead, it focuses on using the segmentation output together with the 2 µm bin-level expression matrix generated by Space Ranger to prepare the inputs required by Bin2Cell and aggregate spatial bins into individual cells.

#### Reference to Bin2Cell

Polański K. *et al.* (2024). **Bin2cell reconstructs cells from high resolution Visium HD data.** *Bioinformatics*, 40(9), btae546.

- Paper: https://doi.org/10.1093/bioinformatics/btae546
- GitHub: https://github.com/Teichlab/bin2cell

## Biological question and Computational Overview

In Visium HD datasets, gene expression is measured on a dense grid of small square bins. Each bin contains transcripts originating from a small region of the tissue, but not necessarily from a single cell.

Bin2Cell uses a segmentation mask derived from a histology or immunofluorescence image to assign bins to individual cells. Once bins have been assigned, the counts from all bins belonging to the same cell are aggregated, producing a **cell × gene** expression matrix.

The conceptual workflow is as follows:

1. Start from a SpatialData object containing the 2 × 2 μm Visium HD expression matrix together with the aligned tissue image.
3. Load a precomputed cell segmentation mask.
4. Convert the segmentation mask into a sparse `.npz` representation, which is more memory-efficient and compatible with Bin2Cell.
5. Assign cell labels to the bin-level AnnData object.
6. Expand cell labels to include cytoplasmic regions.
7. Aggregate bins belonging to the same cell.
8. Perform cell-level quality control and downstream analysis.

## 1. Environment

### 1.1 Import Libraries

The main libraries used in this tutorial are:

- **Scanpy**: for handling AnnData objects, calculating quality-control metrics, and performing downstream single-cell analyses.
- **Bin2Cell**: for assigning spatial bins to segmented cells and generating cell-level expression matrices.
- **SciPy Sparse**: for efficiently storing and loading segmentation masks in sparse `.npz` format.
- **Tifffile**: for reading segmentation masks stored as `.tif` images.
- **Matplotlib** and **Seaborn**: for data visualization and quality-control plots.


In [ ]:
# Standard library utilities for paths, memory, randomness, and warnings.
from pathlib import Path
from datetime import datetime

# Core scientific Python libraries for arrays, dataframes, sparse matrices, and plotting.
import spatialdata as sd
import spatialdata_plot
from spatialdata.models import Labels2DModel, TableModel, ShapesModel
from spatialdata.transformations import Translation, Identity, set_transformation, get_transformation
import numpy as np
import pandas as pd
import scipy as sp
import scipy.sparse
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import seaborn as sns
import tifffile

# Utilities for image loading and processing.
import cv2

# Functions for identifying object boundaries in segmentation masks.
from skimage.segmentation import find_boundaries

# Scanpy and spatial data utilities for single-cell/spatial omics processing.
import scanpy as sc

# Bin2Cell tools for converting Visium HD bin-level data into cell-level matrices.
import bin2cell as b2c

### 1.2 Settings

In [ ]:
sns.set_theme(style="whitegrid")
sc.settings.verbosity = 3

### 1.3 Files and parameters

Modify only this cell to adapt the notebook to your local environment.

#### Main Inputs

- `sdata`: SpatialData object containing the Visium HD gene expression matrix at 2 × 2 μm resolution together with the aligned tissue image.
- `segmentation_mask_tif`: Precomputed cell segmentation mask generated with an external segmentation tool. Each cell is assigned a unique integer ID, while background pixels are encoded as `0`.

#### Main Outputs

- `sdata`: Updated SpatialData object containing all analysis results, including cell assignments, aggregated cell-level expression, and downstream analysis outputs.

In [ ]:
# Inputs

# zarr path to the P5 dataset
zarr_path = Path("/workspaces/data/P5_crop_slide.zarr")
assert zarr_path.exists(), f"Missing SpatialData: {zarr_path}"

# maks with cell segmentation output
segmentation_mask_tif = Path("/workspaces/data/P5_crop_nuclei_masks.tif")
assert segmentation_mask_tif.exists(), f"Missing mask: {segmentation_mask_tif}"


In [ ]:
sample_id = "P5"
resolution = "002um"

In [ ]:
results_dir = Path("/workspaces/results/")
results_dir.mkdir(parents=True, exist_ok=True)

# Sparse segmentation labels
labels_npz_path = results_dir / f"{sample_id}_segmentation_labels_sparse.npz"

# Sparse segmentation labels
final_zarr_path = results_dir / f"{sample_id}_crop_slide_annotated.zarr"

### 1.4 Start computations

In [ ]:
print(datetime.now())

## 2. Preparing the `.npz` File from a Segmentation Mask and Loading it in the Spatialdata

### Load the segmentation mask and convert it to sparse `.npz` format

Bin2Cell expects cell segmentation labels to be provided in a sparse `.npz` format. Compared with a dense `.tif` image, this representation is considerably more memory-efficient and allows large segmentation masks to be handled more efficiently.

The segmentation mask must satisfy the following requirements:

- background pixels must be encoded as `0`;
- each cell must be assigned a unique positive integer identifier (`1`, `2`, `3`, ...);
- the mask must be aligned to the same coordinate system used by the tissue image and Space Ranger output;
- the mask must contain individual cell labels rather than a simple binary segmentation, since Bin2Cell needs to distinguish between different cells.

This section demonstrates **how to generate the sparse `.npz` file required by Bin2Cell** starting from an existing segmentation mask.

In [ ]:
# Load the segmentation mask produced externally.
# Each pixel should contain 0 for background or a positive integer cell ID.
mask = tifffile.imread(segmentation_mask_tif)

print(f"Mask shape: {mask.shape}")
print(f"Mask dtype: {mask.dtype}")
print(f"Minimum label: {mask.min()}")
print(f"Maximum label: {mask.max()}")
print(f"Detected segmentation labels, excluding background 0: {len(np.unique(mask)) - 1}")

In [ ]:
# Save the dense label image as a sparse matrix in NPZ format.
# This is the format used later by b2c.insert_labels().
mask_sparse = scipy.sparse.csr_matrix(mask.astype(np.uint32))
scipy.sparse.save_npz(labels_npz_path, mask_sparse)

print(f"Saved sparse labels NPZ: {labels_npz_path}")
print(f"Sparse matrix shape: {mask_sparse.shape}")
print(f"Sparse matrix non-zero pixels: {mask_sparse.nnz}")


### Loading the SpatialData Object and Registering the Segmentation Mask

Before preparing the segmentation mask for Bin2Cell, we first load the SpatialData object generated from the Space Ranger output. This object contains the Visium HD data, including the tissue image, spatial coordinates, and gene expression matrix.

The segmentation mask produced during the previous tutorial is then imported into the same SpatialData object as a labeled image. During this step, the mask is assigned the appropriate spatial transformation so that it is correctly aligned with the Visium HD coordinate system.

At the end of this section, the SpatialData object contains both the transcriptomics data and the cell segmentation mask within a common spatial reference frame, allowing subsequent processing and visualization.

In [ ]:
# load the dataset using the read_zarr function
sdata_roi = sd.read_zarr(zarr_path)
sdata_roi

In [ ]:
# Mask coordinates in the P5 frame of reference
x_min, x_max = 59000, 63000
y_min, y_max = 55000, 62000

In [ ]:
sdata_roi.labels["cell_segmentation_mask"] = Labels2DModel.parse(
    mask,
    dims=("y", "x"),
    transformations={
        "P5": Translation([59000, 55000], axes=("x", "y"))
    },

)

In [ ]:
sdata_roi

## 3. Quick Visual Inspection of the Segmentation Mask

Before proceeding with Bin2Cell, we visualize the segmentation mask generated in the previous tutorial. 
This allows us to:
- inspect the segmentation results
- verify that the mask was loaded correctly
- confirm that cell boundaries and labels appear as expected before aggregating spatial bins into individual cells.

In [ ]:
sdata_roi.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
).pl.render_labels(
    "cell_segmentation_mask",
    fill_alpha=0,
    outline_alpha=1,
).pl.show(
    coordinate_systems="P5",
    dpi=100
)

To better inspect the cell segmentation results, we focus on a small region of interest (ROI) and adjust the image contrast for visualization. 

This example also demonstrates the basic SpatialData plotting workflow by combining image and label layers.

**In this section, we will:**
- Crop a small ROI from the dataset.
- Estimate robust display intensity limits from a downsampled image.
- Overlay the cell segmentation boundaries on the image, with a plot with increased resolution.

In [ ]:
sdata_roi_crop = sdata_roi.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[60000, 58000],
    max_coordinate=[61000, 59000],
    target_coordinate_system="P5",
)

In [ ]:
img = sdata_roi.images["P5_crop_image"].data
sample = img[:, ::10, ::10]
sample_np = sample.compute()

vmin = np.percentile(sample_np, 1)
vmax = np.percentile(sample_np, 99)

norm = Normalize(vmin=vmin, vmax=vmax)

In [ ]:
sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_labels(
    "cell_segmentation_mask",
    fill_alpha=0,
    outline_alpha=1,
    color="red"
).pl.show(
    coordinate_systems="P5",
    dpi=200
)

### Visualize the AOI segmentation

The `aoi_segmentation_mask` layer is already included in the `SpatialData` object. For simplicity, we pre-loaded this segmentation instead of repeating the integration steps described for the cell segmentation mask.

Although this tutorial does not focus on AOI-level analysis, the same workflow demonstrated throughout the notebook can be applied to these segmented regions.

**In the following cell, we will:**
- Visualize the pre-loaded AOI segmentation.
- Compare it with the cell segmentation shown previously.

In [ ]:
sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_labels(
    "aoi_segmentation_mask",
    fill_alpha=0,
    outline_alpha=1,
    color="red"
).pl.show(
    coordinate_systems="P5",
    dpi=200
)

In [ ]:
del sdata_roi_crop

## 4. Prepare a Bin2Cell-compatible AnnData object

Instead of reading the Space Ranger output with `b2c.read_visium()`, we reconstruct the required `AnnData` object directly from the `SpatialData` container.

We start from the `square_002um` table, which contains the Visium HD 2 μm bin-level expression data. To make this object compatible with the Bin2Cell workflow, we need to adapt the spatial information stored in the `AnnData` object:

**Structures updated in `adata_bins`:**
- `obsm["spatial"]`: stores the x/y coordinates of each 2 μm bin. These coordinates are shifted from the global image reference frame to the local coordinate system of the cropped ROI.
- `uns["spatial"]`: stores image-related metadata in the format expected by Visium-like workflows. Here, we add:
  - the cropped high-resolution image;
  - the spatial scale factors;
  - the spot diameter;
  - minimal image metadata.

Starting from the 2 μm bins preserves the highest spatial resolution available in Visium HD and provides the most accurate input for downstream bin-to-cell assignment.

**In this section, we will:**
- Extract the `square_002um` table from `SpatialData`.
- Convert bin coordinates to the local ROI reference frame.
- Add the image and spatial metadata required by Bin2Cell.

In [ ]:
sdata_roi

In [ ]:
adata_bins = sdata_roi.tables["square_002um"].copy()

coords_xy = adata_bins.obsm["spatial"]

adata_bins.obsm["spatial"] = np.column_stack([
    coords_xy[:, 0] - x_min,
    coords_xy[:, 1] - y_min,
]).astype(float)

adata_bins.uns["spatial"] = {
    sample_id: {
        "images": {
            "hires": sdata_roi.images["P5_crop_image"].copy().transpose("y", "x", "c"),
        },
        "scalefactors": {
            "tissue_hires_scalef": 1.0,
            "spot_diameter_fullres": 2.0,
        },
        "metadata": {
            "source_image_path": "",
        },
    }
}

## 5. Verifying Alignment Between the Image, Spatial Coordinates, and Segmentation Mask

This is a critical quality-control step. The segmentation mask and the Visium HD bin coordinates must be defined in the same coordinate system. If they are misaligned, bins may be assigned to incorrect cells, leading to inaccurate cell-level expression profiles.


At this stage, the goal is to confirm that:

- the segmentation mask and tissue image have compatible dimensions;
- the spatial coordinates of the bins align correctly with the image;
- no scaling, rotation, or coordinate-system mismatches are present;

Performing this check before cell aggregation helps identify potential alignment issues early in the workflow and ensures that the resulting cell-level expression matrix accurately reflects the underlying tissue structure.

In [ ]:
# Check that the mask dimensions are compatible with the image coordinates inferred by Bin2Cell.
b2c.actual_vs_inferred_image_shape(adata_bins, mask)
print("Image/mask dimensions are compatible with the inferred Space Ranger coordinates.")


## 6. Assigning Cell Labels to Spatial Bins

The `insert_labels()` function assigns a cell label to each spatial bin based on the bin coordinates and the segmentation mask stored in the `.npz` file.

After this step, each bin is associated with a cell identifier, allowing Bin2Cell to determine which bins belong to the same segmented cell.

The resulting labels are stored in:

```python
adata_bins.obs["labels"]

In [ ]:
b2c.insert_labels(
    adata=adata_bins,
    labels_npz_path=str(labels_npz_path),
    basis="spatial",
    spatial_key="spatial",
    mpp=None,
    labels_key="labels",
)

assigned_bins = np.sum(adata_bins.obs["labels"] > 0)
unassigned_bins = np.sum(adata_bins.obs["labels"] == 0)
detected_cells = len(np.unique(adata_bins.obs["labels"])) - 1

print("Assignment statistics")
print(f"Total bins: {adata_bins.n_obs}")
print(f"Bins assigned to cells: {assigned_bins}")
print(f"Unassigned bins: {unassigned_bins}")
print(f"Cells detected: {detected_cells}")


## 7. Expanding Cell Labels

Segmentation methods based on nuclear staining often capture only the nucleus, whereas many transcripts are located in the cytoplasm or in the pericellular space. To account for this, Bin2Cell provides a label expansion step that extends the region assigned to each cell beyond the original segmentation boundaries.

The `volume_ratio` parameter controls the extent of this expansion. Choosing an appropriate value is important because it directly affects how transcripts are assigned to cells:

- a value that is too low may fail to capture transcripts located outside the nucleus, resulting in signal loss;
- a value that is too high may assign transcripts from neighboring cells, increasing contamination between cells.

The optimal setting depends on the tissue type, cell density, and the characteristics of the segmentation method used. In practice, moderate expansion often improves transcript recovery while preserving cell-specific expression profiles.

In [ ]:
b2c.expand_labels(
    adata_bins,
    labels_key="labels",
    expanded_labels_key="labels_expanded",
    volume_ratio=3,
)

print(adata_bins.obs[["labels", "labels_expanded"]].head())


In [ ]:
assigned_bins = np.sum(adata_bins.obs["labels_expanded"] > 0)
unassigned_bins = np.sum(adata_bins.obs["labels_expanded"] == 0)
detected_cells = len(np.unique(adata_bins.obs["labels_expanded"])) - 1

print("Assignment statistics")
print(f"Total bins: {adata_bins.n_obs}")
print(f"Bins assigned to cells: {assigned_bins}")
print(f"Unassigned bins: {unassigned_bins}")
print(f"Cells detected: {detected_cells}")


### Add Bin2Cell cell boundaries to the SpatialData object

The Bin2Cell workflow assigns each 2 μm bin to a reconstructed cell through the `labels_expanded` annotation. To integrate these results into the `SpatialData` object, we convert the bin-level assignments into cell-level geometries.

Starting from the existing bin shapes stored in `SpatialData`, we associate each bin with its reconstructed cell, merge all bins belonging to the same cell into a single geometry, and register the resulting cell boundaries as a new `Shapes` element. This representation can then be linked to a cell-level `AnnData` table for downstream spatial transcriptomics analyses.

**In this section, we will:**
- Map the Bin2Cell cell assignments (`labels_expanded`) to the existing 2 μm bin geometries.
- Merge bins assigned to the same cell into cell-level boundaries.
- Apply a small geometric smoothing to improve the visualization of cell outlines.
- Register the reconstructed cell boundaries as a new `Shapes` element in the `SpatialData` object.
- Visualize the reconstructed cell boundaries over the tissue image.

In [ ]:
sdata_roi.shapes["P5_square_002um"]

In [ ]:
str(sdata_roi.shapes["P5_square_002um"].loc[11, "geometry"])

In [ ]:
sdata_roi.shapes["P5_square_002um"].loc[11, "geometry"]

In [ ]:
bin_shapes = sdata_roi.shapes["P5_square_002um"].copy()

label_map = (
    adata_bins.obs
    .set_index("location_id")["labels_expanded"]
)

bin_shapes["cell_id"] = label_map.reindex(bin_shapes.index)

bin_shapes = bin_shapes.dropna(subset=["cell_id"]).copy()
bin_shapes = bin_shapes[bin_shapes["cell_id"] > 0].copy()

cell_boundaries = bin_shapes.dissolve(by="cell_id")

buffer_size = 0.5
cell_boundaries["geometry"] = (
    cell_boundaries.geometry
    .buffer(buffer_size)
    .buffer(-buffer_size)
)

cell_boundaries = ShapesModel.parse(cell_boundaries)

set_transformation(
    cell_boundaries,
    Identity(),
    to_coordinate_system="P5",
)

sdata_roi.shapes["bin2cell_cell_boundaries"] = cell_boundaries

sdata_roi

In [ ]:
sdata_roi.shapes["bin2cell_cell_boundaries"]

In [ ]:
str(sdata_roi.shapes["bin2cell_cell_boundaries"].loc[11, "geometry"])

In [ ]:
sdata_roi.shapes["bin2cell_cell_boundaries"].loc[11, "geometry"]

In [ ]:
sdata_roi_crop = sdata_roi.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[60000, 58000],
    max_coordinate=[61000, 59000],
    target_coordinate_system="P5",
)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="red",
    fill_alpha=0.08,
    outline_color="red",
    outline_width=0.4,
    outline_alpha=0.6,
).pl.show(
    coordinate_systems="P5",
    dpi=200
)

In [ ]:
del sdata_roi_crop

## 8. Aggregating Spatial Bins into Cells

This is the core step of the Bin2Cell workflow. The `bin_to_cell()` function aggregates transcript counts from all bins assigned to the same cell and generates a new AnnData object at the cell level.

The resulting object has the standard structure used in single-cell transcriptomics:

- `adata_cells.obs` contains cell-level metadata;
- `adata_cells.var` contains gene-level information;
- `adata_cells.X` stores the **cell × gene** count matrix.

At this stage, each observation corresponds to a reconstructed cell rather than an individual spatial bin, making the dataset directly compatible with standard single-cell analysis workflows.

In this tutorial, we use the `labels_expanded` assignments rather than the original nuclear labels. In many datasets, label expansion improves transcript recovery by including cytoplasmic and pericellular transcripts that would otherwise be excluded from the reconstructed cell expression profiles.

In [ ]:
adata_cells = b2c.bin_to_cell(
    adata=adata_bins,
    labels_key="labels_expanded",
    spatial_keys=["spatial"],
    diameter_scale_factor=None,
)

print(adata_cells)
print(f"Cells: {adata_cells.n_obs}")
print(f"Genes: {adata_cells.n_vars}")


In [ ]:
adata_cells.obs.head()

## 9. Quality Control of the Cell-Level Matrix

We now treat `adata_cells` similarly to a single-cell RNA-seq dataset.

In [ ]:
sc.set_figure_params()

### Quality metrics

We compute standard quality-control metrics, including:

- `total_counts`: total number of UMIs per cell;
- `n_genes_by_counts`: number of detected genes per cell;
- `pct_counts_mt`: percentage of mitochondrial counts;

These metrics help identify low-quality cells, potential technical artefacts, and regions with an unusually high mitochondrial signal.

In [ ]:
adata_cells.var["gene_symbols_upper"] = adata_cells.var_names.str.upper()
adata_cells.var["mt"] = adata_cells.var["gene_symbols_upper"].str.startswith("MT-")

sc.pp.calculate_qc_metrics(
    adata_cells,
    qc_vars=["mt"],
    inplace=True,
    percent_top=None,
    log1p=False,
)

adata_cells.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].head()

### Defining QC Thresholds for Cell-Level Data

The thresholds used for quality control should be considered dataset-specific and may need to be adjusted depending on tissue type, segmentation quality, sequencing depth, and experimental design.

In this tutorial, the goal is not to identify the optimal filtering criteria, but rather to illustrate the general principles of cell-level quality control.

Typical filtering steps include:

- removing cells with very low total UMI counts;
- removing cells with a low number of detected genes;
- removing cells with an unusually high percentage of mitochondrial counts;


Applying these filters helps reduce technical noise and improves the quality of downstream analyses by retaining cells with sufficient biological information.

In [ ]:
min_counts_cell = 100
min_genes_cell = 50
max_mt_cell = 25

These histograms help guide the choice of QC thresholds. In a real analysis, thresholds should not be selected only a priori, but should also be informed by the distribution of QC metrics within the sample.

Visualizing these metrics allows us to identify low-quality cells, outliers, and potential technical artefacts before applying filtering criteria.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(adata_cells.obs["total_counts"], bins=60, ax=axs[0])
axs[0].axvline(min_counts_cell, color="red", linestyle="--")
axs[0].set_title("Total counts per cell")

sns.histplot(adata_cells.obs["n_genes_by_counts"], bins=60, ax=axs[1])
axs[1].axvline(min_genes_cell, color="red", linestyle="--")
axs[1].set_title("Genes per cell")

sns.histplot(adata_cells.obs["pct_counts_mt"], bins=60, ax=axs[2])
axs[2].axvline(max_mt_cell, color="red", linestyle="--")
axs[2].set_title("% mitochondrial counts")

plt.tight_layout()
plt.show()


We can now check how many cells would be filtered based on the thresholds that we have defined above.

In [ ]:
adata_cells.obs["qc_low_counts"] = adata_cells.obs["total_counts"] < min_counts_cell
adata_cells.obs["qc_low_genes"] = adata_cells.obs["n_genes_by_counts"] < min_genes_cell
adata_cells.obs["qc_high_mt"] = adata_cells.obs["pct_counts_mt"] > max_mt_cell
adata_cells.obs["qc_outlier"] = (adata_cells.obs["qc_low_counts"] | adata_cells.obs["qc_low_genes"] |  adata_cells.obs["qc_high_mt"] )

qc_summary = pd.DataFrame({
    "metric": [
        "Total cells",
        "Low counts",
        "Low genes",
        "High mitochondrial percentage",
        "QC outliers",
        "Cells passing QC",
    ],
    "n_cells": [
        adata_cells.n_obs,
        int(adata_cells.obs["qc_low_counts"].sum()),
        int(adata_cells.obs["qc_low_genes"].sum()),
        int(adata_cells.obs["qc_high_mt"].sum()),
        int(adata_cells.obs["qc_outlier"].sum()),
        int((~adata_cells.obs["qc_outlier"]).sum()),
    ]
}).set_index("metric")

qc_summary

### Filtering the QC-Filtered Cell-Level Object

After defining the quality-control criteria, we create a new AnnData object containing only the cells that pass all QC filters.

Keeping both the filtered and unfiltered datasets is considered good practice. The unfiltered object preserves the original data and allows the filtering strategy to be revisited if different thresholds are required in the future.

The QC-filtered dataset will serve as the starting point for downstream analyses such as normalization, dimensionality reduction, clustering, cell type annotation, and spatial characterization of cellular populations.

In [ ]:
adata_cells_qc = adata_cells[~adata_cells.obs["qc_outlier"].astype(bool)].copy()

print(f"Cells before QC: {adata_cells.n_obs}")
print(f"Cells after QC: {adata_cells_qc.n_obs}")

## 10. Normalization, Feature Selection and Dimensionality Reduction

We first preserve the raw count matrix in `layers['counts']`. Then we normalize each cell to the same total count depth, log-transform the data, select highly variable genes and compute PCA.

This is a standard exploratory workflow. Parameters such as the number of highly variable genes or PCs can be adjusted depending on dataset size and biological complexity.


In [ ]:
# Store raw counts before normalization.
adata_cells_qc.layers["counts"] = adata_cells_qc.X.copy()

# Normalize counts per cell and log-transform.
sc.pp.normalize_total(adata_cells_qc, target_sum=1e4)
sc.pp.log1p(adata_cells_qc)

# Select highly variable genes and compute PCA.
sc.pp.highly_variable_genes(
    adata_cells_qc,
    n_top_genes=3000,
    flavor="seurat",
)
sc.tl.pca(
    adata_cells_qc,
    n_comps=50,
    svd_solver="arpack",
    mask_var="highly_variable"
)

In [ ]:
# Plot variance explained by PCs.
sc.pl.pca_variance_ratio(adata_cells_qc, n_pcs=50, show=False)

In [ ]:
# Plot cells along the PC1 vs PC2 dimensions
sc.pl.pca(adata_cells_qc, color=["total_counts", "n_genes_by_counts", "pct_counts_mt"])

## 11. Neighborhood Graph, UMAP and Leiden Clustering

We now build a nearest-neighbor graph in PCA space, compute a UMAP embedding and run Leiden clustering.

The Leiden resolution controls cluster granularity. Lower values produce fewer, broader clusters; higher values produce more clusters.

In [ ]:
sc.pp.neighbors(
    adata_cells_qc,
    n_neighbors=15,
    n_pcs=30,
)
sc.tl.umap(adata_cells_qc)

In [ ]:
cluster_key = "leiden_0.5"
sc.tl.leiden(
    adata_cells_qc,
    resolution=0.5,
    key_added=cluster_key,
)

In [ ]:
adata_cells_qc.obs[cluster_key].value_counts().sort_index()

In [ ]:
# UMAP colored by Leiden clusters and main QC metrics.
sc.pl.umap(
    adata_cells_qc,
    color=[cluster_key, "total_counts", "n_genes_by_counts", "pct_counts_mt"],
    wspace=0.35,
    ncols=2
)

## 12. Marker genes ad cluster cell-type annotation

Marker genes help interpret the biological identity of clusters. Here we define a small example marker panel covering some common populations.

Only genes present in the dataset are plotted. The marker list should be adapted to the tissue, model system and annotation goals.

In [ ]:
# Example marker panel for neural / brain organoid-like datasets.
marker_genes = {
    "Epithelial": ["EPCAM", "KRT8", "KRT18"],
    "Stem-like": ["LGR5", "ASCL2", "SOX9"],
    "Proliferating": ["MKI67", "TOP2A", "CENPF"],
    "Fibroblasts": ["COL1A1", "DCN", "LUM"],
    "Endothelial": ["PECAM1", "VWF", "KDR"],
    "T cells": ["CD3D", "CD3E", "TRBC2"],
    "B cells": ["MS4A1", "CD79A", "CD79B"],
    "Myeloid": ["LYZ", "FCN1", "S100A8"],
}

# Keep only markers present in the object.
marker_genes_present = {
    group: [gene for gene in genes if gene in adata_cells_qc.var_names]
    for group, genes in marker_genes.items()
}
marker_genes_present = {
    group: genes for group, genes in marker_genes_present.items() if len(genes) > 0
}

print("Markers present in the dataset:")
for group, genes in marker_genes_present.items():
    print(f"{group}: {', '.join(genes)}")


A dotplot summarizes marker expression across clusters:

- dot color represents average expression;
- dot size represents the fraction of cells expressing the gene.

This is useful for assigning tentative biological identities to clusters.


In [ ]:
dotplot = sc.pl.dotplot(
    adata_cells_qc,
    marker_genes_present,
    groupby=cluster_key,
    standard_scale="var",
    dendrogram=False
)

After inspecting UMAPs and marker dotplots, clusters can be assigned biological labels.


In [ ]:
# Manual cluster annotation based on marker gene expression

cluster_annotation = {
    "0": "B cells",
    "1": "Fibroblasts",
    "2": "Fibroblasts",
    "3": "Epithelial cells",
    "4": "Epithelial cells",
    "5": "Epithelial cells",
    "6": "Fibroblasts",
    "7": "Epithelial cells",
    "8": "Epithelial cells",
    "9": "Myeloid cells",
}

adata_cells_qc.obs["manual_annotation"] = (
    adata_cells_qc.obs[cluster_key]
    .map(cluster_annotation)
    .astype("category")
)

print(
    adata_cells_qc.obs[[cluster_key, "manual_annotation"]]
    .drop_duplicates()
    .sort_values(cluster_key)
)



In [ ]:
sc.pl.umap(
    adata_cells_qc,
    color="manual_annotation",
    legend_loc="right margin",
)

## 13. Spatial Visualization of QC metrics, markers and cell annotations

all the annotations that are contained in the anndata object can also visualized spatially by including the anndata in the spatialdata object

### Anndata integrated as a spatialdata table

In [ ]:
sdata_roi

In [ ]:
adata_cells_qc.obs["region"] = "bin2cell_cell_boundaries"

adata_cells_qc = TableModel.parse(
    adata_cells_qc,
    region="bin2cell_cell_boundaries",
    region_key="region",
    instance_key="object_id",
)

In [ ]:
sdata_roi.tables["bin2cell_cells"] = adata_cells_qc

In [ ]:
sdata_roi

In [ ]:
sdata_roi.tables["bin2cell_cells"]

### QC metrics

This step helps determine whether low-quality cells or outliers are randomly distributed across the tissue or concentrated in specific regions. Spatially localized QC issues may indicate tissue damage, poor segmentation, low RNA quality, or technical artefacts affecting particular areas of the section.

In [ ]:
sdata_roi_crop = sdata_roi.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[60000, 58000],
    max_coordinate=[61000, 59000],
    target_coordinate_system="P5",
)

In [ ]:
fig, axs = plt.subplots(
    1, 2,
    figsize=(15, 5),
)

fig.subplots_adjust(wspace=0.8)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="pct_counts_mt",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[0]
)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="total_counts",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[1]
)


### Marker genes

In [ ]:
fig, axs = plt.subplots(
    2, 2,
    figsize=(15, 10),
)

fig.subplots_adjust(wspace=0.8)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="EPCAM",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[0, 0]
)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="LYZ",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[0, 1]
)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="DCN",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[1, 0]
)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="MS4A1",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[1, 1]
)


### Cell Clusters

If spatial coordinates are available in `adata_cells_qc.obsm['spatial']`, we can project the Leiden clusters back onto the tissue section.

This is useful to check whether transcriptional clusters correspond to coherent spatial regions or tissue structures.


In [ ]:
fig, axs = plt.subplots(
    1, 2,
    figsize=(15, 5),
)

fig.subplots_adjust(wspace=0.8)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="leiden_0.5",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[0]
)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="manual_annotation",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[1]
)

In [ ]:
sdata_roi_crop = sdata_roi.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[60000, 55000],
    max_coordinate=[64000, 58000],
    target_coordinate_system="P5",
)

In [ ]:
fig, axs = plt.subplots(
    1, 2,
    figsize=(15, 5),
)

fig.subplots_adjust(wspace=0.8)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="leiden_0.5",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[0]
)

sdata_roi_crop.pl.render_images(
    "P5_crop_image",
    grayscale=True,
    cmap="grey",
    norm=norm,
).pl.render_shapes(
    "bin2cell_cell_boundaries",
    color="manual_annotation",
    table_name="bin2cell_cells",
    fill_alpha=0.5,
    outline_alpha=0.2,
).pl.show(
    coordinate_systems="P5",
    dpi=100,
    ax=axs[1]
)

## 14. Save Processed Spatialdata Object

In [ ]:
del sdata_roi_crop.tables["bin2cell_cells"].uns["spatial"]
del sdata_roi_crop.tables["bin2cell_cells"].obsm["spatial"]

In [ ]:
sdata_roi_crop.write(
    final_zarr_path,
    overwrite=True,
)

## Summary

In this tutorial we :

- generated a sparse .npz segmentation mask compatible with Bin2Cell,
- used Bin2Cell to reconstruct cell-level gene expression profiles,
- generated a QC-filtered cell-level AnnData object;
- computed and visualized quality-control metrics and summary statistics;
- performed downstream analyses, including clustering and spatial visualization of marker gene expression at the cell level.